In [ ]:
# Imports
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image
import numpy as np
import seaborn as sns
import pandas as pd
from scipy import stats, ndimage
import os
import pathlib

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/staining_single_dendrites_ort' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/staining_single_dendrites_ort' 
savepath_fig.mkdir(parents=True, exist_ok=True)

cm_inch = 1/2.54

%config InlineBackend.figure_format = 'retina'

colors = {'Ort': '#e8872c', 'Rdl': '#52c3c3', 'GluCl': '#5f974e', 'Da7': '#204f74'}

In [ ]:
files = [f for f in inputpath_data.rglob("*.tif")]

In [ ]:
intensity = pd.DataFrame(columns=['image', 'celltype', 'receptor', 'number', 'intensity_values'])

for image in files:
    name = image.stem
    celltype, receptor, number = name.split("_")
    intensity_values = np.average(np.array(Image.open(image)), axis=0)
    temp = pd.DataFrame({'image':name, 'celltype':celltype, 'receptor':receptor, 'number':number, 'intensity_values':[intensity_values]})
    intensity = pd.concat([intensity, temp])
intensity = intensity.reset_index(drop=True)
intensity['length'] = intensity.intensity_values.apply(lambda x: len(x))
intensity['interpolated_distance'] = [np.linspace(0, 1, intensity.length.max())]*len(intensity)
intensity['interpolated_intensity'] = intensity.apply(lambda row: np.interp(row.interpolated_distance, np.linspace(0,1,row.length), row.intensity_values), axis=1)
intensity['normalized_intensity'] = intensity.apply(lambda row: (row.interpolated_intensity - row.interpolated_intensity.min()) / (row.interpolated_intensity.max() - row.interpolated_intensity.min()), axis=1)
intensity['smooth_intensity'] = intensity.apply(lambda row: (ndimage.gaussian_filter1d(row.normalized_intensity, sigma=8, mode='constant', cval=0)), axis=1)
intensity['xy_plotting'] = intensity.apply(lambda row: list(zip(row.interpolated_distance, row.smooth_intensity)), axis=1)
intensity['color'] = intensity.receptor.map(colors)
intensity.groupby(['celltype', 'receptor']).number.unique()

In [ ]:
agg_data = pd.DataFrame(intensity.groupby(['celltype', 'receptor']).smooth_intensity.agg(list)).reset_index()
agg_data['mean_intensity'] = agg_data.apply(lambda row: np.stack(row.smooth_intensity).mean(axis=0), axis=1)
agg_data['sem_intensity'] = agg_data.apply(lambda row: stats.sem(np.stack(row.smooth_intensity), axis=0), axis=1)
agg_data['interpolated_distance'] = [np.linspace(0, 1, intensity.length.max())]*len(agg_data)
agg_data['xy_plotting_mean'] = agg_data.apply(lambda row: list(zip(row.interpolated_distance, row.mean_intensity)), axis=1)
agg_data['color'] = agg_data.receptor.map(colors)

In [ ]:
plotting = agg_data[['celltype', 'receptor', 'interpolated_distance', 'mean_intensity', 'sem_intensity', 'color']].copy()
plotting = plotting.explode(['interpolated_distance', 'mean_intensity', 'sem_intensity'])

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(4, 0.8), sharey=True)

subset = plotting.query("celltype == 'T4'")
sns.lineplot(data=subset, x='interpolated_distance', y='mean_intensity', hue_order=['GluCl', 'Da7', 'Rdl', 'Ort'], hue='receptor', legend=False, palette=colors, ax=ax[0], style='receptor', dashes={'Ort': (), 'Rdl': (2,2), 'GluCl': (2,2), 'Da7': (2,2)});
error_ort_T4 = subset.query("receptor == 'Ort'")
ax[0].fill_between(x=error_ort_T4.interpolated_distance.values.astype(float), y1=(error_ort_T4.mean_intensity.values - error_ort_T4.sem_intensity.values).astype(float), y2=(error_ort_T4.mean_intensity.values + error_ort_T4.sem_intensity.values).astype(float), color='#e8872c', alpha=0.2, lw=0);
sns.despine()

subset = plotting.query("celltype == 'T5'")
sns.lineplot(data=subset, x='interpolated_distance', y='mean_intensity', hue_order=['GluCl', 'Da7', 'Rdl', 'Ort'], hue='receptor', palette=colors, ax=ax[1], style='receptor', dashes={'Ort': (), 'Rdl': (2,2), 'GluCl': (2,2), 'Da7': (2,2)});
error_ort_T5 = subset.query("receptor == 'Ort'")
ax[1].fill_between(x=error_ort_T5.interpolated_distance.values.astype(float), y1=(error_ort_T5.mean_intensity.values - error_ort_T5.sem_intensity.values).astype(float), y2=(error_ort_T5.mean_intensity.values + error_ort_T5.sem_intensity.values).astype(float), color='#e8872c', alpha=0.2, lw=0);
ax[1].yaxis.set_visible(False)
ax[1].spines['left'].set_visible(False) 

for axis in ax:
    axis.invert_xaxis()
    axis.set_xlabel('Normalized distance (AU)')
    axis.set_ylabel('Normalized\nfluorescence intensity(AU)')
    axis.set_ylim([0,0.8])
    axis.set_xticks([1, 0], labels=['D', 'P'])
    axis.set_yticks([0, 0.4, 0.8])

plt.margins(x=0.02,y=0)
plt.subplots_adjust(wspace=0.1)

handles = [
    Line2D([0], [0], color=colors['GluCl'], linestyle=(0, (2, 2)), label='GluCl'),
    Line2D([0], [0], color=colors['Da7'],   linestyle=(0, (2, 2)), label='Da7'),
    Line2D([0], [0], color=colors['Rdl'],   linestyle=(0, (2, 2)), label='Rdl'),
    Line2D([0], [0], color=colors['Ort'],   linestyle='-',         label='Ort'),
]

ax[1].legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5))

figtitle = 'ort_fluorescence_intensity.pdf'
plt.savefig(savepath_fig / figtitle)